In [1]:
import json, os
os.makedirs('notebooks', exist_ok=True)

cells = []

def md(src):
    return {"cell_type": "markdown", "metadata": {}, "source": src}

def code(src):
    return {"cell_type": "code", "execution_count": None,
            "metadata": {}, "outputs": [], "source": src}

cells.append(md("# Script 17 — 3D Pose Visualization\n**Pan-coronavirus RTC Inhibitor Discovery** | GIGA-VIN Lab, ULiège | 2026"))

cells.append(code("""import py3Dmol
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE   = Path('..')
POSES  = BASE / '04-hits/poses'
RECEPT = BASE / '03-virtual-screening'

TARGETS = {
    'NSP12-NSP7': {
        'rec': RECEPT / 'NSP12-NSP7_3/receptor_NSP12-NSP7_3.pdb',
        'key_res': [440, 557, 553, 545, 541],
        'color': '#2196F3',
    },
    'NSP9-NSP12': {
        'rec': RECEPT / 'NSP9-NSP12_5/receptor_NSP9-NSP12_5.pdb',
        'key_res': [733, 760, 761, 694, 689],
        'color': '#F44336',
    },
    'NSP12-NSP8': {
        'rec': RECEPT / 'NSP12-NSP8_4/receptor_NSP12-NSP8_4.pdb',
        'key_res': [332, 99, 35, 50, 83],
        'color': '#4CAF50',
    },
}

LEADS = [
    ('351017',   (-8.153, -8.132, -7.759), 'Scaffold A — Lead #1'),
    ('13633807', (-8.081, -8.173, -7.218), 'Scaffold C — Lead #2'),
    ('5024943',  (-8.225, -7.261, -7.790), 'Scaffold B — Lead #5'),
]
print('Setup OK')"""))

cells.append(md("## Helper Functions"))

cells.append(code("""def pdbqt_to_pdb_str(pdbqt_path):
    lines, in_m1 = [], False
    with open(pdbqt_path) as f:
        for line in f:
            if line.startswith('MODEL        1'):
                in_m1 = True
                continue
            if line.startswith('ENDMDL') and in_m1:
                break
            if in_m1 and line.startswith(('ATOM','HETATM')):
                lines.append(line[:66].rstrip())
    return '\\n'.join(lines)

def make_complex(rec_pdb, lig_pdbqt):
    with open(rec_pdb) as f:
        rec = [l.rstrip() for l in f if l.startswith(('ATOM','HETATM','TER'))]
    lig_lines = pdbqt_to_pdb_str(lig_pdbqt).split('\\n')
    lig_out = []
    for line in lig_lines:
        if line.startswith(('ATOM','HETATM')):
            line = 'HETATM' + line[6:17] + 'LIG Z   1' + line[26:66]
        lig_out.append(line)
    return '\\n'.join(rec + ['TER'] + lig_out + ['END'])

def get_center(pdbqt_path):
    coords, in_m1 = [], False
    with open(pdbqt_path) as f:
        for line in f:
            if line.startswith('MODEL        1'):
                in_m1 = True
            if line.startswith('ENDMDL') and in_m1:
                break
            if in_m1 and line.startswith(('ATOM','HETATM')):
                try:
                    coords.append([float(line[30:38]),
                                   float(line[38:46]),
                                   float(line[46:54])])
                except:
                    pass
    return np.mean(coords, axis=0) if coords else np.zeros(3)

def show_pose(zinc_id, target_name, width=900, height=520):
    tinfo    = TARGETS[target_name]
    rec_pdb  = tinfo['rec']
    lig_path = POSES / target_name / f'{zinc_id}_out.pdbqt'
    if not lig_path.exists():
        print(f'Pose not found: {lig_path}')
        return
    cplx   = make_complex(rec_pdb, lig_path)
    center = get_center(lig_path)
    v = py3Dmol.view(width=width, height=height)
    v.addModel(cplx, 'pdb')
    # Receptor cartoon
    v.setStyle({'chain': {'$ne': 'Z'}},
               {'cartoon': {'color': 'lightgrey', 'opacity': 0.9}})
    # Binding pocket surface
    v.addSurface(py3Dmol.SAS,
                 {'opacity': 0.12, 'color': 'lightblue'},
                 {'chain': {'$ne': 'Z'},
                  'within': {'distance': 8, 'sel': {'chain': 'Z'}}})
    # Key residues
    for resi in tinfo['key_res']:
        v.setStyle({'resi': resi, 'chain': {'$ne': 'Z'}},
                   {'cartoon': {'color': 'orange'},
                    'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.18}})
    # Ligand sticks
    v.setStyle({'chain': 'Z'},
               {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.25}})
    # Ligand centroid sphere
    v.addSphere({'center': {'x': float(center[0]),
                            'y': float(center[1]),
                            'z': float(center[2])},
                 'radius': 1.5, 'color': 'yellow', 'opacity': 0.25})
    v.zoomTo({'chain': 'Z'})
    v.setBackgroundColor('white')
    v.show()

print('Helper functions ready')
print('Usage: show_pose(zinc_id, target_name)')"""))

cells.append(md("## Lead #1: ZINC351017 in NSP9–NSP12 (best score: −8.132 kcal/mol)\nNiRAN domain · ARG733 pharmacophore · druggability 0.895"))
cells.append(code("show_pose('351017', 'NSP9-NSP12')"))

cells.append(md("## Lead #1: ZINC351017 in NSP12–NSP7 (score: −8.153 kcal/mol)\nPHE440 aromatic groove · druggability 0.961"))
cells.append(code("show_pose('351017', 'NSP12-NSP7')"))

cells.append(md("## Lead #1: ZINC351017 in NSP12–NSP8 (score: −7.759 kcal/mol)\nLYS332–ASP99 salt bridge network · druggability 0.874"))
cells.append(code("show_pose('351017', 'NSP12-NSP8')"))

cells.append(md("## Lead #2: ZINC13633807 — Scaffold C Benzimidazoquinoxaline"))
cells.append(code("""show_pose('13633807', 'NSP9-NSP12')  # best score -8.173"""))

cells.append(md("## Lead #5: ZINC5024943 — Scaffold B Bicyclic hydrazide\nBest NSP12-NSP7 score in entire screen: −8.225 kcal/mol"))
cells.append(code("""show_pose('5024943', 'NSP12-NSP7')  # best score -8.225"""))

cells.append(md("## Best Single-Target Hit: ZINC5430538 in NSP9–NSP12 (−9.526 kcal/mol)"))
cells.append(code("""import subprocess
from pathlib import Path

p = POSES / 'NSP9-NSP12' / '5430538_out.pdbqt'
if not p.exists():
    print('Downloading from NIC5...')
    subprocess.run([
        'scp',
        'nic5:/scratch/ulg/gigambd/onsekuye/rtc-screening/results/NSP9-NSP12/5430538_out.pdbqt',
        str(POSES / 'NSP9-NSP12/')
    ])
show_pose('5430538', 'NSP9-NSP12')"""))

cells.append(md("## Quick Loop — All 3 Leads in their Best Target"))
cells.append(code("""for zinc_id, scores, label in LEADS:
    best_idx    = scores.index(min(scores))
    best_target = list(TARGETS.keys())[best_idx]
    best_score  = scores[best_idx]
    print(f'ZINC{zinc_id} | {label} | {best_target} | {best_score:.3f} kcal/mol')
    show_pose(zinc_id, best_target, width=800, height=450)"""))

nb = {
    "nbformat": 4,
    "nbformat_minor": 5,
    "metadata": {
        "kernelspec": {
            "display_name": "rtc-discovery",
            "language": "python",
            "name": "rtc-discovery"
        },
        "language_info": {"name": "python", "version": "3.10.0"}
    },
    "cells": cells
}

with open('notebooks/17_3d_pose_visualization.ipynb', 'w') as f:
    json.dump(nb, f, indent=1)

# Validate
with open('notebooks/17_3d_pose_visualization.ipynb') as f:
    json.load(f)

print('Notebook created and validated: notebooks/17_3d_pose_visualization.ipynb')

Notebook created and validated: notebooks/17_3d_pose_visualization.ipynb
